# IPMSM LightGBM Surrogate Model

Train LightGBM surrogate models from the united IPMSM simulation result CSV files.

What this notebook does:
- Uses no wandb dependency.
- Uses only the selected 20 geometry/input columns as model inputs.
- Trains one LightGBM regressor per output target.
- Supports optional random-search hyperparameter tuning by validation RMSE.
- Draws test-set actual-vs-predicted scatter plots with R2, MAE, RMSE, and MAPE.


## Unit Suffix Meaning

The suffix at the end of each output column indicates the unit or statistic.

- `_nm`: torque in Newton-meter (`N*m`)
- `_w`: power or loss in Watt (`W`)
- `_h`: inductance in Henry (`H`)
- `_pct`: percent (`%`)
- `_last_avg`: average over the last electrical period of the transient solution

Note: the requested `output_efficiency_last_pc` column is not present in the CSV files. The actual column name is `output_efficiency_last_pct`, and the notebook maps it automatically.


In [ ]:
from pathlib import Path
import json
import pickle
import random
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    import lightgbm as lgb
except ImportError as exc:
    raise ImportError('Run this notebook with a kernel that has LightGBM installed, for example the Anaconda deap_25v1 environment.') from exc

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

SEED = 42
set_seed(SEED)
print('lightgbm version:', lgb.__version__)


In [ ]:
# =========================
# User configuration
# =========================
PROJECT_DIR = Path(r'Y:/git/pyaedt_motor')
DATA_PATHS = [
    PROJECT_DIR / 'ipmsm_simulation_results1.csv',
    PROJECT_DIR / 'ipmsm_simulation_results2.csv',
]
MODEL_DIR = PROJECT_DIR / 'lightgbm_ipmsm_models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# If the same case_id appears in multiple files, keep the last occurrence.
DROP_DUPLICATE_CASE_ID = True

INPUT_COLUMNS = [
    'input_slot_num',
    'input_pole_num',
    'input_stator_outer_radius',
    'input_stator_back_yoke_thick_ratio',
    'input_stator_back_yoke_thick',
    'input_stator_inner_ratio',
    'input_stator_inner_radius',
    'input_stator_shoe_thick',
    'input_stator_teeth_length_ratio',
    'input_stator_teeth_length',
    'input_stator_teeth_width',
    'input_stator_gap',
    'input_rotator_gap',
    'input_shaft_ratio',
    'input_rotor_radius',
    'input_shaft_radius',
    'input_magnet_shield_thick',
    'input_magnet_setback_ratio',
    'input_magnet_thick_ratio',
    'input_magnet_height_ratio',
]

REQUESTED_OUTPUT_COLUMNS = [
    'output_torque_last_avg_nm',
    'output_torque_last_max_nm',
    'output_total_loss_last_avg_w',
    'output_solidloss_last_avg_w',
    'output_coreloss_last_avg_w',
    'output_ld_last_avg_h',
    'output_lq_last_avg_h',
    'output_efficiency_last_pc',
]
OUTPUT_ALIASES = {'output_efficiency_last_pc': 'output_efficiency_last_pct'}

TEST_SIZE = 0.20
VAL_SIZE = 0.20

# Set True to run random-search tuning per output target using validation RMSE.
ENABLE_TUNING = True
N_TUNING_TRIALS = 100

# Set True to remove output outliers before training for comparison.
REMOVE_OUTPUT_OUTLIERS = True
OUTLIER_IQR_WEIGHT = 1.5


In [ ]:
# =========================
# Load, unite, and validate data
# =========================
missing_files = [path for path in DATA_PATHS if not path.exists()]
if missing_files:
    raise FileNotFoundError(f'Missing data file(s): {missing_files}')

frames = []
for path in DATA_PATHS:
    part = pd.read_csv(path, encoding='utf-8-sig')
    part['source_file'] = path.name
    frames.append(part)
    ok_count = part['status'].astype(str).str.lower().eq('ok').sum() if 'status' in part.columns else 'n/a'
    print(f'Loaded: {path} | shape={part.shape} | ok={ok_count}')

raw_df = pd.concat(frames, ignore_index=True, sort=False)
before_dedup = len(raw_df)
if DROP_DUPLICATE_CASE_ID and 'case_id' in raw_df.columns:
    raw_df = raw_df.drop_duplicates(subset='case_id', keep='last').reset_index(drop=True)
    print(f'Dropped duplicate case_id rows: {before_dedup - len(raw_df)}')

print(f'United raw shape: {raw_df.shape}')

OUTPUT_COLUMNS = []
OUTPUT_NAME_MAP = {}
for requested_col in REQUESTED_OUTPUT_COLUMNS:
    actual_col = requested_col if requested_col in raw_df.columns else OUTPUT_ALIASES.get(requested_col, requested_col)
    if actual_col not in raw_df.columns:
        raise ValueError(f'Missing output column: requested={requested_col}, actual={actual_col}')
    OUTPUT_COLUMNS.append(actual_col)
    OUTPUT_NAME_MAP[requested_col] = actual_col

missing_inputs = [col for col in INPUT_COLUMNS if col not in raw_df.columns]
if missing_inputs:
    raise ValueError(f'Missing input columns: {missing_inputs}')

df = raw_df.copy()
for col in INPUT_COLUMNS + OUTPUT_COLUMNS:
    df[col] = pd.to_numeric(df[col], errors='coerce')

status_ok = df['status'].astype(str).str.lower().eq('ok') if 'status' in df.columns else pd.Series(True, index=df.index)
finite_inputs = np.isfinite(df[INPUT_COLUMNS]).all(axis=1)
finite_outputs = np.isfinite(df[OUTPUT_COLUMNS]).all(axis=1)
valid_mask = status_ok & finite_inputs & finite_outputs
valid_df = df.loc[valid_mask].copy()

if REMOVE_OUTPUT_OUTLIERS:
    keep = pd.Series(True, index=valid_df.index)
    for col in OUTPUT_COLUMNS:
        q1 = valid_df[col].quantile(0.25)
        q3 = valid_df[col].quantile(0.75)
        iqr = q3 - q1
        low = q1 - OUTLIER_IQR_WEIGHT * iqr
        high = q3 + OUTLIER_IQR_WEIGHT * iqr
        keep &= valid_df[col].between(low, high)
    before = len(valid_df)
    valid_df = valid_df.loc[keep].copy()
    print(f'Removed output outliers: {before - len(valid_df)} row(s)')

print('Output column mapping:')
for requested_col, actual_col in OUTPUT_NAME_MAP.items():
    print(f'  {requested_col} -> {actual_col}')
print(f'Valid rows: {len(valid_df):,} / {len(raw_df):,}')
valid_df[INPUT_COLUMNS + OUTPUT_COLUMNS].describe().T


In [ ]:
# =========================
# Train/validation/test split
# =========================
if len(valid_df) < 20:
    raise RuntimeError(f'Valid data is too small for robust training: {len(valid_df)} rows')

X = valid_df[INPUT_COLUMNS].copy()
Y = valid_df[OUTPUT_COLUMNS].copy()

X_train_val, X_test, Y_train_val, Y_test = train_test_split(
    X, Y, test_size=TEST_SIZE, random_state=SEED
)
relative_val_size = VAL_SIZE / (1.0 - TEST_SIZE)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train_val, Y_train_val, test_size=relative_val_size, random_state=SEED
)
print('X_train:', X_train.shape, 'X_val:', X_val.shape, 'X_test:', X_test.shape)
print('Y_train:', Y_train.shape, 'Y_val:', Y_val.shape, 'Y_test:', Y_test.shape)


In [ ]:
# =========================
# LightGBM helpers
# =========================
BASE_PARAMS = {
    'objective': 'regression',
    'n_estimators': 800,
    'learning_rate': 0.03,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 10,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'reg_alpha': 0.0,
    'reg_lambda': 0.1,
    'random_state': SEED,
    'n_jobs': -1,
    'verbosity': -1,
}

PARAM_SEARCH_SPACE = {
    'n_estimators': [400, 700, 1000, 1400],
    'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08],
    'num_leaves': [15, 31, 63, 127],
    'max_depth': [-1, 4, 6, 8, 10],
    'min_child_samples': [5, 10, 20, 40],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.7, 0.85, 1.0],
    'reg_alpha': [0.0, 1e-3, 1e-2, 0.1],
    'reg_lambda': [0.0, 1e-3, 1e-2, 0.1, 1.0],
}

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    r2 = r2_score(y_true, y_pred)
    denom = np.where(np.abs(y_true) < 1e-12, np.nan, np.abs(y_true))
    mape = np.nanmean(np.abs((y_true - y_pred) / denom)) * 100.0
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE_pct': mape}

def build_model(params):
    merged = dict(BASE_PARAMS)
    merged.update(params)
    return lgb.LGBMRegressor(**merged)

def predict_model(model, x):
    best_iter = getattr(model, 'best_iteration_', None)
    if best_iter is not None and best_iter > 0:
        return model.predict(x, num_iteration=best_iter)
    return model.predict(x)

def fit_lgbm(target_col, params):
    model = build_model(params)
    model.fit(
        X_train,
        Y_train[target_col],
        eval_set=[(X_val, Y_val[target_col])],
        eval_metric='l2',
        callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(0)],
    )
    return model

def sample_params(rng):
    return {key: rng.choice(values) for key, values in PARAM_SEARCH_SPACE.items()}

def tune_params_for_target(target_col, n_trials=N_TUNING_TRIALS):
    rng = random.Random(SEED + abs(hash(target_col)) % 100000)
    best_params = dict(BASE_PARAMS)
    best_rmse = float('inf')
    records = []
    for trial in range(1, n_trials + 1):
        params = sample_params(rng)
        model = fit_lgbm(target_col, params)
        val_pred = predict_model(model, X_val)
        metric = regression_metrics(Y_val[target_col], val_pred)
        record = dict(params)
        record.update({'target': target_col, 'trial': trial, 'val_RMSE': metric['RMSE'], 'val_R2': metric['R2']})
        records.append(record)
        if metric['RMSE'] < best_rmse:
            best_rmse = metric['RMSE']
            best_params = params
    return best_params, pd.DataFrame(records).sort_values('val_RMSE')

def train_one_target(target_col):
    if ENABLE_TUNING:
        best_params, tuning_df = tune_params_for_target(target_col)
    else:
        best_params = dict(BASE_PARAMS)
        tuning_df = pd.DataFrame()

    model = fit_lgbm(target_col, best_params)
    rows = []
    predictions = {}
    for split_name, split_X, split_y in [
        ('train', X_train, Y_train[target_col]),
        ('val', X_val, Y_val[target_col]),
        ('test', X_test, Y_test[target_col]),
    ]:
        pred = predict_model(model, split_X)
        predictions[split_name] = pred
        metric = regression_metrics(split_y, pred)
        metric.update({'target': target_col, 'split': split_name, 'best_iteration': getattr(model, 'best_iteration_', None)})
        rows.append(metric)
    return model, best_params, pd.DataFrame(rows), predictions, tuning_df


In [ ]:
# =========================
# Train one LightGBM model per output target
# =========================
models = {}
best_params_by_target = {}
all_metrics = []
all_predictions = {}
tuning_records = []

for target_col in OUTPUT_COLUMNS:
    print(f'Training: {target_col} | tuning={ENABLE_TUNING}')
    model, best_params, metric_df, predictions, tuning_df = train_one_target(target_col)
    models[target_col] = model
    best_params_by_target[target_col] = best_params
    all_metrics.append(metric_df)
    all_predictions[target_col] = predictions
    if not tuning_df.empty:
        tuning_records.append(tuning_df)

metrics_df = pd.concat(all_metrics, ignore_index=True)
metrics_df = metrics_df[['target', 'split', 'MAE', 'RMSE', 'R2', 'MAPE_pct', 'best_iteration']]
tuning_df = pd.concat(tuning_records, ignore_index=True) if tuning_records else pd.DataFrame()
metrics_df


In [ ]:
# Test split performance summary
test_metrics = metrics_df[metrics_df['split'].eq('test')].sort_values('target')
test_metrics


In [ ]:
# Show top tuning trials per target when tuning is enabled.
if ENABLE_TUNING and not tuning_df.empty:
    display(tuning_df.groupby('target').head(5))
else:
    print('Tuning is disabled. Set ENABLE_TUNING = True and rerun training cells to tune hyperparameters.')


In [ ]:
# =========================
# Test-set scatter plots with metrics
# =========================
plot_dir = MODEL_DIR / 'plots'
plot_dir.mkdir(parents=True, exist_ok=True)

def metric_text(metrics):
    return '\n'.join([
        f"R2: {metrics['R2']:.4f}",
        f"MAE: {metrics['MAE']:.4g}",
        f"RMSE: {metrics['RMSE']:.4g}",
        f"MAPE: {metrics['MAPE_pct']:.2f}%",
    ])

def plot_test_scatter(target_col, ax=None, save=True):
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(7, 7))
    y_true = Y_test[target_col].to_numpy(dtype=float)
    y_pred = all_predictions[target_col]['test']
    metrics = regression_metrics(y_true, y_pred)
    min_val = float(min(np.min(y_true), np.min(y_pred)))
    max_val = float(max(np.max(y_true), np.max(y_pred)))
    pad = (max_val - min_val) * 0.05 if max_val > min_val else 1.0
    min_plot = min_val - pad
    max_plot = max_val + pad

    ax.scatter(y_true, y_pred, alpha=0.68, edgecolor='k', linewidth=0.35)
    ax.plot([min_plot, max_plot], [min_plot, max_plot], 'r--', lw=2, label='Ideal')
    ax.set_xlim(min_plot, max_plot)
    ax.set_ylim(min_plot, max_plot)
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(target_col)
    ax.grid(True, alpha=0.3)
    ax.text(
        0.05, 0.95, metric_text(metrics), transform=ax.transAxes,
        va='top', ha='left', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.78),
    )
    if own_fig:
        fig.tight_layout()
        if save:
            safe_name = target_col.replace('/', '_').replace(' ', '_')
            out_path = plot_dir / f'{safe_name}_test_scatter.png'
            fig.savefig(out_path, dpi=170)
            print('saved:', out_path)
        plt.show()
    return metrics

scatter_metrics = []
for target_col in OUTPUT_COLUMNS:
    m = plot_test_scatter(target_col)
    scatter_metrics.append({'target': target_col, **m})
scatter_metrics_df = pd.DataFrame(scatter_metrics)
scatter_metrics_df


In [ ]:
# Grid plot containing all target scatter plots.
n_targets = len(OUTPUT_COLUMNS)
n_cols = 3
n_rows = int(np.ceil(n_targets / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5.5 * n_rows))
axes = np.array(axes).reshape(-1)
for ax, target_col in zip(axes, OUTPUT_COLUMNS):
    plot_test_scatter(target_col, ax=ax, save=False)
for ax in axes[n_targets:]:
    ax.axis('off')
fig.tight_layout()
grid_path = plot_dir / 'all_targets_test_scatter_grid.png'
fig.savefig(grid_path, dpi=170)
print('saved:', grid_path)
plt.show()


In [ ]:
# =========================
# Save models and metadata
# =========================
model_paths = {}
for target_col, model in models.items():
    safe_name = target_col.replace('/', '_').replace(' ', '_')
    model_path = MODEL_DIR / f'{safe_name}_lgbm.pkl'
    with model_path.open('wb') as f:
        pickle.dump(model, f)
    model_paths[target_col] = str(model_path)

metrics_path = MODEL_DIR / 'metrics.csv'
metrics_df.to_csv(metrics_path, index=False, encoding='utf-8-sig')
if not tuning_df.empty:
    tuning_df.to_csv(MODEL_DIR / 'tuning_trials.csv', index=False, encoding='utf-8-sig')

metadata = {
    'data_paths': [str(path) for path in DATA_PATHS],
    'drop_duplicate_case_id': bool(DROP_DUPLICATE_CASE_ID),
    'source_files': sorted(raw_df['source_file'].dropna().unique().tolist()) if 'source_file' in raw_df.columns else [],
    'input_columns': INPUT_COLUMNS,
    'requested_output_columns': REQUESTED_OUTPUT_COLUMNS,
    'actual_output_columns': OUTPUT_COLUMNS,
    'output_name_map': OUTPUT_NAME_MAP,
    'valid_rows': int(len(valid_df)),
    'raw_rows': int(len(raw_df)),
    'enable_tuning': bool(ENABLE_TUNING),
    'n_tuning_trials': int(N_TUNING_TRIALS),
    'best_params_by_target': best_params_by_target,
    'model_paths': model_paths,
}
metadata_path = MODEL_DIR / 'metadata.json'
with metadata_path.open('w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print('Saved model dir:', MODEL_DIR)
print('Saved metrics:', metrics_path)
print('Saved metadata:', metadata_path)


In [ ]:
# =========================
# Prediction helper for NSGA-II
# =========================
def predict_outputs(input_df):
    """Predict all trained output targets from a dataframe containing INPUT_COLUMNS."""
    missing = [col for col in INPUT_COLUMNS if col not in input_df.columns]
    if missing:
        raise ValueError(f'Missing input columns for prediction: {missing}')
    x = input_df[INPUT_COLUMNS].copy()
    for col in INPUT_COLUMNS:
        x[col] = pd.to_numeric(x[col], errors='coerce')
    if not np.isfinite(x).all(axis=None):
        raise ValueError('Input dataframe contains NaN or infinite values.')

    pred = pd.DataFrame(index=input_df.index)
    for target_col, model in models.items():
        pred[target_col] = predict_model(model, x)
    return pred

# Example: predict first five valid rows
example_pred = predict_outputs(valid_df.head(5))
example_pred
